In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install segmentation-models-pytorch albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.8 MB/s eta 0:00:00


In [3]:
import os

# 1. Nạp Token và Tải Data
os.environ['KAGGLE_API_TOKEN'] = "KGAT_eee51873186b5d8faffb87b5bbf56416"

print("Đang tải dữ liệu từ Kaggle. Có thể mất vài phút...")
# Lệnh tải bộ dữ liệu BRISC từ Kaggle
!kaggle datasets download -d briscdataset/brisc2025

# Lệnh giải nén dữ liệu vào thư mục brisc_data
!unzip -q brisc2025.zip -d brisc_data
print("✅ Hoàn tất tải và giải nén!")

os.environ['BRISC_DATA_ROOT'] = '/content/brisc_data/brisc2025'

# 2. Kiểm tra cấu trúc
dest = '/content/brisc_data/brisc2025'
expected = [
    os.path.join(dest, "classification_task", "train"),
    os.path.join(dest, "segmentation_task",   "train", "images"),
    os.path.join(dest, "segmentation_task",   "train", "masks"),
]

print("\n── Kiểm tra đường dẫn ──")
ok = True
for path in expected:
    exists = os.path.isdir(path)
    symbol = "✓" if exists else "✗"
    print(f"  {symbol} {path}")
    if not exists: ok = False

if ok:
    print("\n✓ Cấu trúc chuẩn xác! Có thể chuyển qua bước Train.")
else:
    print("\n⚠ Cảnh báo: Sai cấu trúc! Hãy thử kiểm tra lệnh !ls /content/brisc_data")




Đang tải dữ liệu từ Kaggle. Có thể mất vài phút...
Dataset URL: https://www.kaggle.com/datasets/briscdataset/brisc2025
License(s): Attribution 4.0 International (CC BY 4.0)
100% 250M/250M [00:02<00:00, 87.8MB/s]

✅ Hoàn tất tải và giải nén!

── Kiểm tra đường dẫn ──
  ✓ /content/brisc_data/brisc2025/classification_task/train
  ✓ /content/brisc_data/brisc2025/segmentation_task/train/images
  ✓ /content/brisc_data/brisc2025/segmentation_task/train/masks

✓ Cấu trúc chuẩn xác! Có thể chuyển qua bước Train.


In [4]:
!ls -l /content/brisc_data

total 4
drwxr-xr-x 4 root root 4096 Jun 12 04:07 brisc2025


In [5]:
import json
import numpy as np
from copy import deepcopy
import cv2
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm import tqdm

import random

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [6]:
# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
DATA_ROOT = os.environ.get("BRISC_DATA_ROOT", "/content/brisc_data")

SEG_TRAIN_IMG  = os.path.join(DATA_ROOT, "segmentation_task", "train", "images")
SEG_TRAIN_MASK = os.path.join(DATA_ROOT, "segmentation_task", "train", "masks")
SEG_TEST_IMG   = os.path.join(DATA_ROOT, "segmentation_task", "test",  "images")
SEG_TEST_MASK  = os.path.join(DATA_ROOT, "segmentation_task", "test",  "masks")

OUTPUT_DIR = "/content/drive/MyDrive/brisc_project/outputs/segmentation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEG_CLASSES = ["glioma", "meningioma", "pituitary"]
SEG_IMAGE_SIZE   = 256
SEG_BATCH_SIZE   = 16
LEARNING_RATE    = 1e-4
WEIGHT_DECAY     = 1e-5
NUM_WORKERS      = 2  # Giảm xuống 2 để phù hợp với Colab Free

In [7]:
# ==============================================================================
# 2. METRICS
# ==============================================================================
EPS = 1e-7

def iou_score(pred: torch.Tensor, target: torch.Tensor, threshold: float = 0.5) -> float:
    pred   = (torch.sigmoid(pred) > threshold).float()
    target = target.float()
    inter  = (pred * target).sum()
    union  = pred.sum() + target.sum() - inter
    return ((inter + EPS) / (union + EPS)).item()

def dice_score(pred: torch.Tensor, target: torch.Tensor, threshold: float = 0.5) -> float:
    pred   = (torch.sigmoid(pred) > threshold).float()
    target = target.float()
    inter  = (pred * target).sum()
    return ((2 * inter + EPS) / (pred.sum() + target.sum() + EPS)).item()

class SegmentationMeter:
    """Tích lũy IoU và Dice qua toàn bộ test set rồi tính mean."""
    def __init__(self):
        self.ious  = []
        self.dices = []

    @torch.no_grad()
    def update(self, pred, target):
        self.ious.append(iou_score(pred, target))
        self.dices.append(dice_score(pred, target))

    def result(self) -> dict:
        return {
            "mIoU":  float(np.mean(self.ious)),
            "mDice": float(np.mean(self.dices)),
        }

    def reset(self):
        self.ious  = []
        self.dices = []

def print_seg_table(results: dict):
    print(f"\n{'='*70}")
    print(f"  Segmentation Results")
    print(f"{'='*70}")
    header = f"  {'Model':<16} {'Glioma':>10} {'Meningioma':>12} {'Pituitary':>10} {'W-mIoU':>10}"
    print(header)
    print(f"  {'-'*58}")
    for model, v in results.items():
        g  = v.get("glioma",      {}).get("mIoU", 0.0)
        m  = v.get("meningioma",  {}).get("mIoU", 0.0)
        p  = v.get("pituitary",   {}).get("mIoU", 0.0)
        w  = v.get("weighted_mIoU", 0.0)
        print(f"  {model:<16} {g*100:>10.1f} {m*100:>12.1f} {p*100:>10.1f} {w*100:>10.1f}")
    print()

In [8]:
# ==============================================================================
# 3. MODELS
# ==============================================================================
"""
Segmentation Model Factory
============================
Table 3 models:
  • UNet, UNet++, MANet, LinkNet, DeepLabV3+, PAN  → segmentation-models-pytorch
  • EINet, EU-Net, DAD, BASNet, SaberNet, ABANet   → custom implementations

Gọi: model = build_seg_model("UNet")
"""

# ─── SMP-based models ────────────────────────────────────────────────────────

SMP_REGISTRY = {
    "UNet":       ("Unet",         "resnet34"),
    "UNet++":     ("UnetPlusPlus", "resnet34"),
    "MANet":      ("MAnet",        "resnet34"),
    "LinkNet":    ("Linknet",      "resnet34"),
    "DeepLabV3+": ("DeepLabV3Plus","resnet34"),
    "PAN":        ("PAN",          "resnet34"),
}


def _build_smp_model(arch: str, encoder: str,
                     in_channels: int = 3, classes: int = 1) -> nn.Module:
    constructor = getattr(smp, arch)
    return constructor(
        encoder_name=encoder,
        encoder_weights="imagenet",
        in_channels=in_channels,
        classes=classes,
        activation=None,          # raw logits; BCEWithLogitsLoss
    )


# ─── Custom: EU-Net (Enhanced U-Net) ─────────────────────────────────────────
# Patel et al. 2021 – feature-enhancement block inserted at each decoder stage

class _FeatureEnhancement(nn.Module):
    """Squeeze-and-Excitation feature enhancement block."""
    def __init__(self, ch, reduction=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch, max(ch // reduction, 1)),
            nn.ReLU(inplace=True),
            nn.Linear(max(ch // reduction, 1), ch),
            nn.Sigmoid(),
        )

    def forward(self, x):
        w = self.se(x).view(x.size(0), x.size(1), 1, 1)
        return x * w


class _ConvBNReLU(nn.Sequential):
    def __init__(self, in_ch, out_ch, k=3, p=1):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, k, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )


class EUNet(nn.Module):
    """Simplified EU-Net for binary tumor segmentation."""

    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        b = base
        # Encoder
        self.enc1 = nn.Sequential(_ConvBNReLU(in_ch, b), _ConvBNReLU(b, b))
        self.enc2 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b, b*2), _ConvBNReLU(b*2, b*2))
        self.enc3 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b*2, b*4), _ConvBNReLU(b*4, b*4))
        self.enc4 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b*4, b*8), _ConvBNReLU(b*8, b*8))
        # Bottleneck
        self.bottleneck = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b*8, b*16), _ConvBNReLU(b*16, b*16))
        # Decoder with FE
        self.up4  = nn.ConvTranspose2d(b*16, b*8, 2, 2)
        self.fe4  = _FeatureEnhancement(b*16)
        self.dec4 = nn.Sequential(_ConvBNReLU(b*16, b*8), _ConvBNReLU(b*8, b*8))
        self.up3  = nn.ConvTranspose2d(b*8, b*4, 2, 2)
        self.fe3  = _FeatureEnhancement(b*8)
        self.dec3 = nn.Sequential(_ConvBNReLU(b*8, b*4), _ConvBNReLU(b*4, b*4))
        self.up2  = nn.ConvTranspose2d(b*4, b*2, 2, 2)
        self.fe2  = _FeatureEnhancement(b*4)
        self.dec2 = nn.Sequential(_ConvBNReLU(b*4, b*2), _ConvBNReLU(b*2, b*2))
        self.up1  = nn.ConvTranspose2d(b*2, b, 2, 2)
        self.fe1  = _FeatureEnhancement(b*2)
        self.dec1 = nn.Sequential(_ConvBNReLU(b*2, b), _ConvBNReLU(b, b))
        self.head = nn.Conv2d(b, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b  = self.bottleneck(e4)
        d4 = self.dec4(self.fe4(torch.cat([self.up4(b), e4], 1)))
        d3 = self.dec3(self.fe3(torch.cat([self.up3(d4), e3], 1)))
        d2 = self.dec2(self.fe2(torch.cat([self.up2(d3), e2], 1)))
        d1 = self.dec1(self.fe1(torch.cat([self.up1(d2), e1], 1)))
        return self.head(d1)


# ─── Custom: EINet (Pyramid Vision Transformer backbone) ─────────────────────
# Li & Jiao 2022 – simplified version with conv pyramid

class _PyramidBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.b1 = _ConvBNReLU(ch, ch, k=1, p=0)
        self.b3 = _ConvBNReLU(ch, ch)
        self.b5 = _ConvBNReLU(ch, ch, k=5, p=2)
        self.proj = nn.Conv2d(ch * 3, ch, 1)

    def forward(self, x):
        return self.proj(torch.cat([self.b1(x), self.b3(x), self.b5(x)], 1))


class EINet(nn.Module):
    """EINet – simplified pyramid feature network for segmentation."""

    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        b = base
        self.stem  = nn.Sequential(_ConvBNReLU(in_ch, b), _ConvBNReLU(b, b))
        self.down1 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b, b*2), _PyramidBlock(b*2))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b*2, b*4), _PyramidBlock(b*4))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b*4, b*8), _PyramidBlock(b*8))
        self.up3   = nn.ConvTranspose2d(b*8, b*4, 2, 2)
        self.fuse3 = _ConvBNReLU(b*8, b*4)
        self.up2   = nn.ConvTranspose2d(b*4, b*2, 2, 2)
        self.fuse2 = _ConvBNReLU(b*4, b*2)
        self.up1   = nn.ConvTranspose2d(b*2, b, 2, 2)
        self.fuse1 = _ConvBNReLU(b*2, b)
        self.head  = nn.Conv2d(b, out_ch, 1)

    def forward(self, x):
        s  = self.stem(x)
        d1 = self.down1(s)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        u3 = self.fuse3(torch.cat([self.up3(d3), d2], 1))
        u2 = self.fuse2(torch.cat([self.up2(u3), d1], 1))
        u1 = self.fuse1(torch.cat([self.up1(u2), s],  1))
        return self.head(u1)


# ─── Custom: BASNet (Boundary-Aware Segmentation Network) ────────────────────
# Qin et al. 2021 – simplified RSU encoder + boundary refinement

class _RSU(nn.Module):
    """Simplified Residual U-block."""
    def __init__(self, in_ch, mid_ch, out_ch, depth=4):
        super().__init__()
        self.conv_in = _ConvBNReLU(in_ch, out_ch)
        downs, ups = [], []
        ch = out_ch
        for _ in range(depth):
            downs.append(_ConvBNReLU(ch, mid_ch))
            ups.insert(0, _ConvBNReLU(mid_ch * 2, ch))
            ch = mid_ch
        self.downs = nn.ModuleList(downs)
        self.ups   = nn.ModuleList(ups)
        self.pool  = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x0 = self.conv_in(x)
        skips, h = [], x0
        for d in self.downs:
            h = d(h)
            skips.append(h)
            h = self.pool(h)
        for u, s in zip(self.ups, reversed(skips)):
            h = F.interpolate(h, size=s.shape[-2:], mode="bilinear", align_corners=False)
            h = u(torch.cat([h, s], 1))
        return h + x0


class BASNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        self.stage1 = _RSU(in_ch, 16, 64,  depth=7)
        self.pool12 = nn.MaxPool2d(2)
        self.stage2 = _RSU(64,    16, 128, depth=6)
        self.pool23 = nn.MaxPool2d(2)
        self.stage3 = _RSU(128,   16, 256, depth=5)
        self.pool34 = nn.MaxPool2d(2)
        self.stage4 = _RSU(256,   16, 512, depth=4)

        self.up4  = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = _ConvBNReLU(512, 256)
        self.up3  = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = _ConvBNReLU(256, 128)
        self.up2  = nn.ConvTranspose2d(128,  64, 2, 2)
        self.dec1 = _ConvBNReLU(128,  64)
        self.head = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        s1 = self.stage1(x)
        s2 = self.stage2(self.pool12(s1))
        s3 = self.stage3(self.pool23(s2))
        s4 = self.stage4(self.pool34(s3))
        d3 = self.dec3(torch.cat([self.up4(s4), s3], 1))
        d2 = self.dec2(torch.cat([self.up3(d3), s2], 1))
        d1 = self.dec1(torch.cat([self.up2(d2), s1], 1))
        return self.head(d1)


# ─── Custom: DAD (Difference-Aware Decoder) ──────────────────────────────────
# Li et al. 2025 – difference features between encoder levels

class _DifferenceBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = _ConvBNReLU(ch * 2, ch)

    def forward(self, high, low):
        """high: upsampled feature; low: skip connection."""
        diff = torch.abs(high - low)
        return self.conv(torch.cat([high + low, diff], 1))


class DAD(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        b = base
        self.enc1 = nn.Sequential(_ConvBNReLU(in_ch, b),   _ConvBNReLU(b, b))
        self.enc2 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b, b*2),  _ConvBNReLU(b*2, b*2))
        self.enc3 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b*2, b*4),_ConvBNReLU(b*4, b*4))
        self.enc4 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b*4, b*8),_ConvBNReLU(b*8, b*8))
        self.bot  = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b*8, b*16),_ConvBNReLU(b*16, b*16))

        self.up4   = nn.ConvTranspose2d(b*16, b*8, 2, 2)
        self.diff4 = _DifferenceBlock(b*8)
        self.up3   = nn.ConvTranspose2d(b*8,  b*4, 2, 2)
        self.diff3 = _DifferenceBlock(b*4)
        self.up2   = nn.ConvTranspose2d(b*4,  b*2, 2, 2)
        self.diff2 = _DifferenceBlock(b*2)
        self.up1   = nn.ConvTranspose2d(b*2,  b,   2, 2)
        self.diff1 = _DifferenceBlock(b)
        self.head  = nn.Conv2d(b, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x);  e2 = self.enc2(e1)
        e3 = self.enc3(e2); e4 = self.enc4(e3)
        b  = self.bot(e4)
        d4 = self.diff4(self.up4(b),  e4)
        d3 = self.diff3(self.up3(d4), e3)
        d2 = self.diff2(self.up2(d3), e2)
        d1 = self.diff1(self.up1(d2), e1)
        return self.head(d1)


# ─── Custom: SaberNet (Multi-scale Transformer) ──────────────────────────────
# Saber et al. 2025 – multi-scale attention + transformer bottleneck

class _TransformerBlock(nn.Module):
    def __init__(self, dim, heads=4, mlp_ratio=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio), nn.GELU(),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, x):
        B, C, H, W = x.shape
        seq = x.flatten(2).transpose(1, 2)             # (B, HW, C)
        att, _ = self.attn(self.norm1(seq), self.norm1(seq), self.norm1(seq))
        seq = seq + att
        seq = seq + self.mlp(self.norm2(seq))
        return seq.transpose(1, 2).view(B, C, H, W)


class SaberNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        b = base
        self.enc1 = nn.Sequential(_ConvBNReLU(in_ch, b),   _ConvBNReLU(b, b))
        self.enc2 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b, b*2),  _ConvBNReLU(b*2, b*2))
        self.enc3 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b*2, b*4),_ConvBNReLU(b*4, b*4))
        self.enc4 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b*4, b*8),_ConvBNReLU(b*8, b*8))
        self.bot  = nn.Sequential(
            nn.MaxPool2d(2),
            _ConvBNReLU(b*8, b*16),
            _TransformerBlock(b*16, heads=4),
            _ConvBNReLU(b*16, b*16),
        )
        self.up4  = nn.ConvTranspose2d(b*16, b*8, 2, 2)
        self.dec4 = nn.Sequential(_ConvBNReLU(b*16, b*8), _TransformerBlock(b*8, heads=2))
        self.up3  = nn.ConvTranspose2d(b*8, b*4, 2, 2)
        self.dec3 = _ConvBNReLU(b*8, b*4)
        self.up2  = nn.ConvTranspose2d(b*4, b*2, 2, 2)
        self.dec2 = _ConvBNReLU(b*4, b*2)
        self.up1  = nn.ConvTranspose2d(b*2, b, 2, 2)
        self.dec1 = _ConvBNReLU(b*2, b)
        self.head = nn.Conv2d(b, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(e1)
        e3 = self.enc3(e2); e4 = self.enc4(e3)
        b  = self.bot(e4)
        d4 = self.dec4(torch.cat([self.up4(b), e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.head(d1)


# ─── Custom: ABANet (Attention Boundary-Aware Network) ───────────────────────
# Rezvani et al. 2024

class _ChannelAttention(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(
            nn.Linear(ch, max(ch // r, 1)), nn.ReLU(),
            nn.Linear(max(ch // r, 1), ch), nn.Sigmoid()
        )

    def forward(self, x):
        a = self.fc(self.avg(x).flatten(1))
        m = self.fc(self.max(x).flatten(1))
        w = (a + m).view(x.size(0), x.size(1), 1, 1)
        return x * w


class _SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3)
        self.sig  = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(1, keepdim=True)
        mx, _ = x.max(1, keepdim=True)
        return x * self.sig(self.conv(torch.cat([avg, mx], 1)))


class _CBAM(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.ca = _ChannelAttention(ch)
        self.sa = _SpatialAttention()

    def forward(self, x):
        return self.sa(self.ca(x))


class ABANet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        b = base
        self.enc1 = nn.Sequential(_ConvBNReLU(in_ch, b),   _ConvBNReLU(b, b),   _CBAM(b))
        self.enc2 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b, b*2),  _ConvBNReLU(b*2, b*2),  _CBAM(b*2))
        self.enc3 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b*2, b*4),_ConvBNReLU(b*4, b*4),  _CBAM(b*4))
        self.enc4 = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b*4, b*8),_ConvBNReLU(b*8, b*8),  _CBAM(b*8))
        self.bot  = nn.Sequential(nn.MaxPool2d(2), _ConvBNReLU(b*8, b*16),_ConvBNReLU(b*16, b*16))

        self.up4  = nn.ConvTranspose2d(b*16, b*8, 2, 2)
        self.dec4 = nn.Sequential(_ConvBNReLU(b*16, b*8), _CBAM(b*8))
        self.up3  = nn.ConvTranspose2d(b*8, b*4, 2, 2)
        self.dec3 = nn.Sequential(_ConvBNReLU(b*8, b*4), _CBAM(b*4))
        self.up2  = nn.ConvTranspose2d(b*4, b*2, 2, 2)
        self.dec2 = nn.Sequential(_ConvBNReLU(b*4, b*2), _CBAM(b*2))
        self.up1  = nn.ConvTranspose2d(b*2, b, 2, 2)
        self.dec1 = nn.Sequential(_ConvBNReLU(b*2, b), _CBAM(b))
        self.head = nn.Conv2d(b, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(e1)
        e3 = self.enc3(e2); e4 = self.enc4(e3)
        b  = self.bot(e4)
        d4 = self.dec4(torch.cat([self.up4(b), e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.head(d1)


# ─── Unified Factory ─────────────────────────────────────────────────────────

CUSTOM_REGISTRY = {
    "EU-Net":   EUNet,
    "EINet":    EINet,
    "BASNet":   BASNet,
    "DAD":      DAD,
    "SaberNet": SaberNet,
    "ABANet":   ABANet,
}


def build_seg_model(name: str, in_channels: int = 3, classes: int = 1) -> nn.Module:
    """
    Args:
        name : model name – xem SMP_REGISTRY hoặc CUSTOM_REGISTRY
        in_channels : số kênh ảnh đầu vào (3 với RGB)
        classes     : số class output (1 cho binary segmentation)
    """
    if name in SMP_REGISTRY:
        arch, encoder = SMP_REGISTRY[name]
        return _build_smp_model(arch, encoder, in_channels, classes)
    elif name in CUSTOM_REGISTRY:
        return CUSTOM_REGISTRY[name](in_ch=in_channels, out_ch=classes)
    else:
        raise ValueError(f"Unknown segmentation model: '{name}'. "
                         f"Available: {list(SMP_REGISTRY) + list(CUSTOM_REGISTRY)}")


ALL_SEG_MODELS = list(SMP_REGISTRY.keys()) + list(CUSTOM_REGISTRY.keys())


In [9]:
# ==============================================================================
# 4. DATASET & AUGMENTATION
# ==============================================================================
def _get_transforms(image_size, split):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if split == "train":
        return A.Compose([
            A.Resize(image_size, image_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),
            A.Rotate(limit=15, p=0.4),
            A.RandomBrightnessContrast(p=0.3),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])

TUMOR_CODE = {"glioma": "gl", "meningioma": "me", "pituitary": "pi"}

class TumorSegDataset(Dataset):
    """Dataset chỉ chứa ảnh của một loại u."""

    def __init__(self, img_dir, mask_dir, tumor_class, split="train", image_size=256):
        self.transform = _get_transforms(image_size, split)
        code = TUMOR_CODE[tumor_class]

        img_files = {
            os.path.splitext(f)[0]: f
            for f in os.listdir(img_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png")) and f"_{code}_" in f
        }
        mask_files = {
            os.path.splitext(f)[0]: f
            for f in os.listdir(mask_dir)
            if f.lower().endswith(".png") and f"_{code}_" in f
        }
        common = sorted(set(img_files) & set(mask_files))
        self.pairs = [(os.path.join(img_dir, img_files[k]),
                       os.path.join(mask_dir, mask_files[k])) for k in common]

        if not self.pairs:
            # Fallback: nếu không tìm được theo code, dùng toàn bộ
            img_all  = sorted(f for f in os.listdir(img_dir)  if f.lower().endswith((".jpg",".jpeg",".png")))
            mask_all = sorted(f for f in os.listdir(mask_dir) if f.lower().endswith(".png"))
            img_base  = {os.path.splitext(f)[0]: f for f in img_all}
            mask_base = {os.path.splitext(f)[0]: f for f in mask_all}
            common_fb = sorted(set(img_base) & set(mask_base))
            self.pairs = [(os.path.join(img_dir, img_base[k]),
                           os.path.join(mask_dir, mask_base[k])) for k in common_fb]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        ip, mp = self.pairs[idx]
        img  = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.uint8)
        aug  = self.transform(image=img, mask=mask)
        return aug["image"], aug["mask"].unsqueeze(0).float()



In [10]:
# ==============================================================================
# 5. TRAINING LOGIC
# ==============================================================================
class DiceBCELoss(nn.Module):
    def __init__(self, bce_weight: float = 0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, pred, target):
        bce  = self.bce(pred, target)
        pred_sig = torch.sigmoid(pred)
        inter    = (pred_sig * target).sum(dim=(2, 3))
        dice     = 1 - (2 * inter + 1) / (pred_sig.sum(dim=(2,3)) + target.sum(dim=(2,3)) + 1)
        return self.bce_weight * bce + (1 - self.bce_weight) * dice.mean()

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for imgs, masks in tqdm(loader, desc="  train", leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss  = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate_seg(model, loader, device, criterion=None) -> dict:
    model.eval()
    meter = SegmentationMeter()
    total_loss = 0.0
    for imgs, masks in tqdm(loader, desc="  eval ", leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        if criterion is not None:
            loss = criterion(preds, masks)
            total_loss += loss.item()
        meter.update(preds, masks)

    results = meter.result()
    if criterion is not None:
        results["loss"] = total_loss / len(loader)
    return results

# ─── Single run for one model × one tumor class ───────────────────────────────

def run_single_seg(model_name, tumor_class, epochs, batch_size, device, output_dir, image_size=256, val_ratio=0.15):
    # dataset
    full_train_ds = TumorSegDataset(SEG_TRAIN_IMG, SEG_TRAIN_MASK, tumor_class, "train", image_size)
    test_ds  = TumorSegDataset(SEG_TEST_IMG,  SEG_TEST_MASK,  tumor_class, "test",  image_size)

    val_size = int(len(full_train_ds) * val_ratio)
    train_size = len(full_train_ds) - val_size

    train_ds, val_ds = random_split(
        full_train_ds,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    print(f"    Train samples: {len(train_ds)}")
    print(f"    Val samples:   {len(val_ds)}")
    print(f"    Test samples:  {len(test_ds)}")

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_dl   = DataLoader(val_ds,  batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    # model
    model     = build_seg_model(model_name).to(device)
    criterion = DiceBCELoss(bce_weight=0.5)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    # output dir
    save_dir = os.path.join(output_dir, model_name, tumor_class)
    os.makedirs(save_dir, exist_ok=True)

    # history
    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_mIoU": [],
        "val_mDice": [],
        "lr": [] }
    best_iou = -1.0
    best_state = deepcopy(model.state_dict())

    # training loop
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_dl, criterion, optimizer, device)
        scheduler.step()

        val_metrics = evaluate_seg(model, val_dl, device, criterion)
        lr = optimizer.param_groups[0]["lr"]

        print(
            f" [{tumor_class}] "
            f"epoch {epoch:>3}/{epochs} | "
            f"train loss {train_loss:.4f} | "
            f"val loss {val_metrics['loss']:.4f} | "
            f"mIoU {val_metrics['mIoU']:.4f} | "
            f"mDice {val_metrics['mDice']:.4f}"
        )

        # save history
        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_mIoU"].append(val_metrics["mIoU"])
        history["val_mDice"].append(val_metrics["mDice"])
        history["lr"].append(lr)

        # best model
        if val_metrics["mIoU"] > best_iou:
          best_iou = val_metrics["mIoU"]
          best_state = deepcopy(model.state_dict())
          torch.save( best_state, os.path.join(save_dir, "best_model.pt") )

    if best_state is not None:
        model.load_state_dict(best_state)
    test_result = evaluate_seg(model, test_dl, device)

    #  SAVE HISTORY CSV
    history_df = pd.DataFrame(history)
    history_csv_path = os.path.join( save_dir, "training_history.csv" )
    history_df.to_csv(history_csv_path, index=False)

    # PLOT LEARNING CURVES
    # ---- LOSS CURVE ----
    plt.figure(figsize=(8, 5))
    plt.plot( history["epoch"], history["train_loss"], label="Train Loss" )
    plt.plot( history["epoch"], history["val_loss"], label="Val Loss" )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} - {tumor_class} Loss Curve")
    plt.legend()

    loss_plot_path = os.path.join( save_dir, "loss_curve.png" )
    plt.savefig(loss_plot_path)
    plt.close()

    # ---- mIoU CURVE ----
    plt.figure(figsize=(8, 5))
    plt.plot( history["epoch"], history["val_mIoU"], label="Validation mIoU" )
    plt.xlabel("Epoch")
    plt.ylabel("mIoU")
    plt.title(f"{model_name} - {tumor_class} mIoU Curve")
    plt.legend()
    miou_plot_path = os.path.join( save_dir, "miou_curve.png" )
    plt.savefig(miou_plot_path)
    plt.close()

    # SAVE FINAL RESULTS
    final_result_path = os.path.join( save_dir, "final_metrics.json" )
    with open(final_result_path, "w") as f:
        json.dump(test_result, f, indent=2)
    print( f"\n FINAL TEST RESULT | " f"mIoU={test_result['mIoU']:.4f} | " f"mDice={test_result['mDice']:.4f}" )

    return test_result

TUMOR_COUNTS = {"glioma": 254, "meningioma": 306, "pituitary": 300}

def weighted_miou(per_class: dict) -> float:
    total = sum(TUMOR_COUNTS.values())
    w_iou = sum(TUMOR_COUNTS[tc] / total * per_class[tc]["mIoU"] for tc in TUMOR_COUNTS if tc in per_class)
    return w_iou

In [11]:
# ==============================================================================
# 6. CẤU HÌNH CHẠY TRÊN COLAB (SỬA Ở ĐÂY TRƯỚC KHI CHẠY)
# ==============================================================================

if __name__ == "__main__":

    # -------- ĐIỀU CHỈNH CÁC THÔNG SỐ NÀY --------
    MODEL_TO_RUN   = "ABANet"  # Chọn 1 trong 12: "UNet", "UNet++", "MANet", "LinkNet", "DeepLabV3+", "PAN",
                                              # "EU-Net", "EINet", "BASNet", "DAD", "SaberNet", "ABANet"
    TUMOR_CLASS    = None    # Chọn: "glioma", "meningioma", "pituitary", hoặc None (để chạy cả 3)
    EPOCHS         = 50       # Đặt = 2 để chạy test, = 50 để train thật
    # ---------------------------------------------

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Đang sử dụng thiết bị: {device.type.upper()}")

    models_to_run  = [MODEL_TO_RUN] if MODEL_TO_RUN else list(SMP_REGISTRY.keys())
    classes_to_run = [TUMOR_CLASS] if TUMOR_CLASS else SEG_CLASSES
    all_results = {}

    for model_name in models_to_run:
        print(f"\n{'─'*60}\n  MODEL: {model_name}\n{'─'*60}")
        per_class = {}

        for tc in classes_to_run:
            print(f"\n  → Đang huấn luyện cho u: {tc}")
            try:
                result = run_single_seg(
                    model_name=model_name, tumor_class=tc, epochs=EPOCHS,
                    batch_size=SEG_BATCH_SIZE, device=device,
                    output_dir=OUTPUT_DIR, image_size=SEG_IMAGE_SIZE
                )
                per_class[tc] = result
                print(f"  ✓ {tc} hoàn tất: mIoU={result['mIoU']*100:.1f}%")
            except Exception as e:
                print(f"  ✗ {tc} gặp lỗi: {e}")

        if per_class:
            wmIoU = weighted_miou(per_class)
            summary = {tc: v for tc, v in per_class.items()}
            summary["weighted_mIoU"] = wmIoU
            all_results[model_name]  = summary

            out_path = os.path.join(OUTPUT_DIR, f"{model_name}_results.json")
            with open(out_path, "w") as f:
                json.dump({k: (v if isinstance(v, float) else v) for k, v in summary.items()}, f, indent=2, default=float)

    print_seg_table(all_results)
    summary_path = os.path.join(OUTPUT_DIR, "all_results.json")
    with open(summary_path, "w") as f:
        json.dump(all_results, f, indent=2, default=float)
    print(f"\n✅ Xong! Kết quả và Checkpoint đã được lưu tại thư mục: {OUTPUT_DIR}")

🚀 Đang sử dụng thiết bị: CUDA

────────────────────────────────────────────────────────────
  MODEL: ABANet
────────────────────────────────────────────────────────────

  → Đang huấn luyện cho u: glioma
    Train samples: 975
    Val samples:   172
    Test samples:  254


 [glioma] epoch   1/50 | train loss 0.7404 | val loss 0.7027 | mIoU 0.0020 | mDice 0.0039


 [glioma] epoch   2/50 | train loss 0.6785 | val loss 0.6634 | mIoU 0.0376 | mDice 0.0719


 [glioma] epoch   3/50 | train loss 0.6418 | val loss 0.6265 | mIoU 0.1942 | mDice 0.3227


 [glioma] epoch   4/50 | train loss 0.6138 | val loss 0.6043 | mIoU 0.3074 | mDice 0.4665


...

 [pituitary] epoch  47/50 | train loss 0.0727 | val loss 0.0919 | mIoU 0.7913 | mDice 0.8832


 [pituitary] epoch  48/50 | train loss 0.0724 | val loss 0.0928 | mIoU 0.7910 | mDice 0.8830


 [pituitary] epoch  49/50 | train loss 0.0723 | val loss 0.0922 | mIoU 0.7912 | mDice 0.8831


 [pituitary] epoch  50/50 | train loss 0.0719 | val loss 0.0920 | mIoU 0.7914 | mDice 0.8833



 FINAL TEST RESULT | mIoU=0.7996 | mDice=0.8878
  ✓ pituitary hoàn tất: mIoU=80.0%

  Segmentation Results
  Model                Glioma   Meningioma  Pituitary     W-mIoU
  ----------------------------------------------------------
  ABANet                 65.6         90.7       80.0       79.6


✅ Xong! Kết quả và Checkpoint đã được lưu tại thư mục: /content/drive/MyDrive/brisc_project/outputs/segmentation
